<a href="https://colab.research.google.com/github/Aldiss1/MPPM_Aldi/blob/main/JS09/JS09-Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 10.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 23.6 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520956 sha256=d1675337dc9d5bce2d074b6ac5b879af20577c749dc512a6d5895111310f6458
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679066 sha256=f83e1879fe61eb4be1afc81122b09f6676246b795513ecb79417d01c61a200a7
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f5

# Tugas Lab 2
Perbandingan ANNOY, FAISS, HNSW pada Dataset Micro Spotify


Lakukan percobaan penggunaan ANNOY, FAISS, dan HNSWLIB pada dataset sekunder berukuran besar (Micro Spotify) pada link berikut: <https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data>.

* Download data dan load CSV filenya (pilih dataset yg pertama dari dua dataset).
* Pilih hanya fitur numerik saja, dan lakukan normalisasi menggunakan **StandardScaler.**
* Lakukan pencarian track terdekat dan bandingkan hasilnya.
* Lakkan perbandingan pada exact NN, ANNOY, FAISS, dan HNSW

## Boilerplate

Berikut kode dasar yang dapat Anda manfaatkan.

In [2]:
# -------------------------------
# Cek & unduh otomatis dataset Spotify
# -------------------------------
import os, glob, shutil

TARGET = 'spotify_songs.csv'   # nama file yang dipakai kode di bawah

def _cari_csv_dengan_fitur(folder='.'):
    """Cari file CSV yang memiliki kolom audio features Spotify."""
    import pandas as pd  # dipakai di sini agar cell ini bisa jalan lebih dulu
    kolom = {'danceability', 'energy', 'valence', 'tempo'}
    for path in glob.glob(os.path.join(folder, '**', '*.csv'), recursive=True):
        try:
            head = pd.read_csv(path, nrows=0)
        except Exception:
            continue
        if kolom.issubset(set(c.lower() for c in head.columns)):
            return path
    return None

if not os.path.exists(TARGET):
    found = _cari_csv_dengan_fitur('.')
    if found:
        shutil.copy(found, TARGET)
        print('Ditemukan & disiapkan:', found, '->', TARGET)
    else:
        try:
            import kagglehub
            path = kagglehub.dataset_download('bwandowando/spotify-songs-with-attributes-and-lyrics')
            csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
            if csv_files:
                csv_files.sort(key=os.path.getsize, reverse=True)
                shutil.copy(csv_files[0], TARGET)
                print('Diunduh & disiapkan:', csv_files[0], '->', TARGET)
            else:
                print('Tidak ada CSV di hasil unduhan.')
        except Exception as e:
            print('Gagal unduh otomatis:', e)
            print('Silakan unduh manual dari:')
            print('https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data')
            print('lalu upload file CSV-nya ke folder ini (nama bebas).')
else:
    print('File sudah ada:', TARGET)

print()
print('File CSV tersedia:', [f for f in os.listdir('.') if f.endswith('.csv')])

Ditemukan & disiapkan: ./spotify_songs_clean.csv -> spotify_songs.csv

File CSV tersedia: ['spotify_songs.csv', 'spotify_songs_clean.csv']


In [3]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
# pakai file bersih jika tersedia (kolom lyrics dibuang), jika tidak pakai file asli
_path = 'spotify_songs_clean.csv' if os.path.exists('spotify_songs_clean.csv') else 'spotify_songs.csv'
df = pd.read_csv(_path, on_bad_lines='skip', low_memory=False)  # ganti path sesuai lokasi file

# -------------------------------
# Samakan nama kolom (dataset ini pakai 'name' & 'artists')
# Supaya kode modul yang memakai 'track_name'/'track_artist' tetap jalan.
# -------------------------------
if 'track_name' not in df.columns and 'name' in df.columns:
    df['track_name'] = df['name']
if 'track_artist' not in df.columns and 'artists' in df.columns:
    df['track_artist'] = df['artists']
if 'track_album_name' not in df.columns and 'album_name' in df.columns:
    df['track_album_name'] = df['album_name']

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

## Perbandingan Exact NN, ANNOY, FAISS, dan HNSW

In [4]:
# Gunakan satu track sebagai query
X_f32 = X_scaled.astype('float32')
query = X_f32[0]   # ambil track pertama sebagai query point
d = X_f32.shape[1]

In [5]:
# -------------------------------
# 1. Exact NN
# -------------------------------
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_f32)

start = time.time()
dist_exact, idx_exact = nn.kneighbors(query.reshape(1, -1))
time_exact = time.time() - start

print("=== Exact NN ===")
print("Indices:", idx_exact[0])
print("Waktu:", time_exact, "detik")

=== Exact NN ===
Indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Waktu: 0.12727832794189453 detik


In [6]:
# -------------------------------
# 2. ANNOY
# -------------------------------
ann_index = AnnoyIndex(d, 'euclidean')
for i, v in enumerate(X_f32):
    ann_index.add_item(i, v)
ann_index.build(10)

start = time.time()
idx_annoy = ann_index.get_nns_by_vector(query, k)
time_annoy = time.time() - start

print("=== ANNOY ===")
print("Indices:", idx_annoy)
print("Waktu:", time_annoy, "detik")

=== ANNOY ===
Indices: [0, 764272, 749223, 833164, 676733, 523698, 85956, 763625, 906004, 385544]
Waktu: 0.0005574226379394531 detik


In [7]:
# -------------------------------
# 3. FAISS
# -------------------------------
faiss_index = faiss.IndexFlatL2(d)
faiss_index.add(X_f32)

start = time.time()
D_faiss, I_faiss = faiss_index.search(query.reshape(1, -1), k)
time_faiss = time.time() - start

print("=== FAISS ===")
print("Indices:", I_faiss[0])
print("Waktu:", time_faiss, "detik")

=== FAISS ===
Indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Waktu: 0.045255184173583984 detik


In [8]:
# -------------------------------
# 4. HNSW
# -------------------------------
hnsw_index = hnswlib.Index(space='l2', dim=d)
hnsw_index.init_index(max_elements=X_f32.shape[0], ef_construction=200, M=16)
hnsw_index.add_items(X_f32)
hnsw_index.set_ef(50)

start = time.time()
labels_hnsw, dist_hnsw = hnsw_index.knn_query(query.reshape(1, -1), k=k)
time_hnsw = time.time() - start

print("=== HNSW ===")
print("Indices:", labels_hnsw[0])
print("Waktu:", time_hnsw, "detik")

=== HNSW ===
Indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Waktu: 0.0003867149353027344 detik


In [9]:
# -------------------------------
# Ringkasan perbandingan
# -------------------------------
print("Track query:", df.iloc[0]['track_name'], "-", df.iloc[0]['track_artist'])
print()
print("Exact NN terdekat:")
for i in idx_exact[0][:5]:
    print("  -", df.iloc[i]['track_name'], "|", df.iloc[i]['track_artist'])
print()
print("ANNOY terdekat:")
for i in idx_annoy[:5]:
    print("  -", df.iloc[i]['track_name'], "|", df.iloc[i]['track_artist'])
print()
print("FAISS terdekat:")
for i in I_faiss[0][:5]:
    print("  -", df.iloc[i]['track_name'], "|", df.iloc[i]['track_artist'])
print()
print("HNSW terdekat:")
for i in labels_hnsw[0][:5]:
    print("  -", df.iloc[i]['track_name'], "|", df.iloc[i]['track_artist'])

Track query: ! - HELLYEAH

Exact NN terdekat:
  - ! | HELLYEAH
  - Intro (Who The Fuck Are You') | C-Lekktor
  - Testimonium | Lacrimosa
  - Ultimate Satisfaction - The Field Remix | Bear In Heaven
  - Sæglópur | Sigur Rós

ANNOY terdekat:
  - ! | HELLYEAH
  - Testimonium | Lacrimosa
  - Sæglópur | Sigur Rós
  - Turn from Heaven | Therion
  - See You When I Die | BOYO

FAISS terdekat:
  - ! | HELLYEAH
  - Intro (Who The Fuck Are You') | C-Lekktor
  - Testimonium | Lacrimosa
  - Ultimate Satisfaction - The Field Remix | Bear In Heaven
  - Sæglópur | Sigur Rós

HNSW terdekat:
  - ! | HELLYEAH
  - Intro (Who The Fuck Are You') | C-Lekktor
  - Testimonium | Lacrimosa
  - Ultimate Satisfaction - The Field Remix | Bear In Heaven
  - Sæglópur | Sigur Rós


In [10]:
# -------------------------------
# Tabel waktu
# -------------------------------
summary = pd.DataFrame({
    "Metode": ["Exact NN", "ANNOY", "FAISS", "HNSW"],
    "Waktu Query (s)": [time_exact, time_annoy, time_faiss, time_hnsw],
})
summary

,Metode,Waktu Query (s)
0,Exact NN,0.127278
1,ANNOY,0.000557
2,FAISS,0.045255
3,HNSW,0.000387
